# 🎬 Netflix Titles — Data Cleaning Notebook

**Dataset:** `netflix_titles.csv` — 8,807 rows × 12 columns  
**Goal:** Produce a clean, analysis-ready DataFrame.

---

## Table of Contents
1. Load & Profile
2. Fix Misplaced Values (Rating ↔ Duration swap)
3. Handle Missing Values
4. Fix Data Types
5. Strip Whitespace
6. Standardise / Normalise Values
7. Final Verification & Export

---
> **What we deliberately did NOT do** is noted after every section.

## Step 0 — Imports

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

RAW_PATH    = 'netflix_titles.csv'   # change to your local path if needed
OUTPUT_PATH = 'netflix_titles_clean.csv'

---
## Step 1 — Load & Profile

**Why:** Before touching anything, we need a baseline snapshot:
- exact shape
- column data types
- null counts per column
- duplicate row count

This drives every decision that follows.

In [2]:
df = pd.read_csv(RAW_PATH)

print('Shape:', df.shape)
df.head(3)

Shape: (8807, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmmaker Kirst..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thabang Molaba, ...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town teen sets o..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabiha Akkari,...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Action & Adve...","To protect his family from a powerful drug lord, skilled..."


In [3]:
print('=== dtypes ===')
print(df.dtypes)
print()
print('=== Null counts ===')
print(df.isnull().sum())
print()
print('=== Duplicate full rows ===')
print(df.duplicated().sum())
print()
print('=== Duplicate show_id ===')
print(df['show_id'].duplicated().sum())

=== dtypes ===
show_id         object
type            object
title           object
director        object
cast            object
country         object
date_added      object
release_year     int64
rating          object
duration        object
listed_in       object
description     object
dtype: object

=== Null counts ===
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

=== Duplicate full rows ===
0

=== Duplicate show_id ===
0


In [4]:
# Snapshot all unique ratings to catch anomalies
print('Unique rating values:')
print(sorted(df['rating'].dropna().unique()))

Unique rating values:
['66 min', '74 min', '84 min', 'G', 'NC-17', 'NR', 'PG', 'PG-13', 'R', 'TV-14', 'TV-G', 'TV-MA', 'TV-PG', 'TV-Y', 'TV-Y7', 'TV-Y7-FV', 'UR']


**Findings from profiling:**

| Column | Nulls | Notes |
|---|---|---|
| `director` | 2634 | Legitimately unknown for many TV Shows |
| `cast` | 825 | Some titles have no credited cast |
| `country` | 831 | Missing origin info |
| `date_added` | 10 | Small gap |
| `rating` | 4 | + 3 rows where a duration was placed here by mistake |
| `duration` | 3 | Same 3 rows — data was shifted into wrong column |
| `date_added` | object | Should be parsed to datetime |
| No duplicate rows | ✅ | Nothing to drop |

---
## Step 2 — Fix Misplaced Values (Rating ↔ Duration column swap)

**Why:** Three rows (all Louis C.K. specials) have a minute-value like `"74 min"` in `rating` and `NaN` in `duration`. This is a data-entry error — the value was placed in the wrong column. If left as-is:
- Filtering by `rating` will return wrong results.
- Aggregating `duration` will silently miss three rows.

**How we detect it:** Any value in `rating` that ends with `" min"` or `" Season"` belongs in `duration`.

**What we do NOT do:** We do NOT drop these rows — the data is valid, just misplaced.

In [5]:
# Identify rows where duration accidentally landed in rating
mask_shifted = df['rating'].str.match(r'^\d+\s*(min|Season)', na=False) #r is for regex, rege
print(f'Rows with misplaced rating value: {mask_shifted.sum()}')
df[mask_shifted][['show_id', 'title', 'rating', 'duration']]

Rows with misplaced rating value: 3


,show_id,title,rating,duration
5541,s5542,Louis C.K. 2017,74 min,NaN
5794,s5795,Louis C.K.: Hilarious,84 min,NaN
5813,s5814,Louis C.K.: Live at the Comedy Store,66 min,NaN


In [6]:
# Move the value from rating → duration, set rating → NaN (will be handled in Step 3)
df.loc[mask_shifted, 'duration'] = df.loc[mask_shifted, 'rating']
df.loc[mask_shifted, 'rating']   = np.nan

# Verify
df.loc[mask_shifted, ['title', 'rating', 'duration']]

,title,rating,duration
5541,Louis C.K. 2017,NaN,74 min
5794,Louis C.K.: Hilarious,NaN,84 min
5813,Louis C.K.: Live at the Comedy Store,NaN,66 min


---
## Step 3 — Handle Missing Values

Missing value strategy is **column-by-column** because one size never fits all:

| Column | Strategy | Reason |
|---|---|---|
| `director` | Fill with `'Unknown'` | 2634 nulls — directors aren't always listed, especially for TV Shows. Dropping rows would lose 30 % of data. |
| `cast` | Fill with `'Unknown'` | Same logic. |
| `country` | Fill with `'Unknown'` | Preserve row; country can be researched later. |
| `date_added` | Fill with `'Unknown'` | Only 10 rows; we cannot reliably impute a date. |
| `rating` | Fill with `'Not Rated'` | Standard industry label for unrated content. |
| `duration` | Leave as-is for now | Only 3 rows and after Step 2 they are now fixed — check count again. |

**What we do NOT do:**
- We do **not** drop rows with nulls — that would lose valid content records.
- We do **not** impute director/cast names with a mode or mean — that would be factually wrong.
- We do **not** forward-fill dates — temporal proximity means nothing here.

In [7]:
print('Null counts before fill:')
print(df[['director','cast','country','date_added','rating','duration']].isnull().sum())

Null counts before fill:
director      2634
cast           825
country        831
date_added      10
rating           7
duration         0
dtype: int64


In [8]:
fill_map = {
    'director'   : 'Unknown',
    'cast'       : 'Unknown',
    'country'    : 'Unknown',
    'date_added' : 'Unknown',
    'rating'     : 'Not Rated',
}
df.fillna(fill_map, inplace=True)

print('Null counts after fill:')
print(df.isnull().sum())

Null counts after fill:
show_id         0
type            0
title           0
director        0
cast            0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
description     0
dtype: int64


---
## Step 4 — Fix Data Types

**Why:** `date_added` is stored as a plain string like `"September 25, 2021"`. Keeping it as `object` means:
- You can't sort chronologically.
- You can't compute year/month trends.
- Filters like `df[df['date_added'] > '2020-01-01']` silently give wrong results.

We parse it to `datetime64` and set `errors='coerce'` so rows with `'Unknown'` (Step 3 fill) become `NaT` instead of crashing.

**What we do NOT do:** We do NOT parse `duration` into numeric — it mixes `"90 min"` and `"2 Seasons"` which belong in two separate derived columns. That is feature engineering, not cleaning.

In [9]:
df['date_added'] = pd.to_datetime(df['date_added'], format='%B %d, %Y', errors='coerce')

print(df['date_added'].dtype)
print(df['date_added'].isnull().sum(), 'rows could not be parsed (were Unknown or blank)')
df[['title', 'date_added']].head(5)

datetime64[ns]
98 rows could not be parsed (were Unknown or blank)


,title,date_added
0,Dick Johnson Is Dead,2021-09-25
1,Blood & Water,2021-09-24
2,Ganglands,2021-09-24
3,Jailbirds New Orleans,2021-09-24
4,Kota Factory,2021-09-24


---
## Step 5 — Strip Whitespace

**Why:** Hidden leading/trailing spaces are invisible but break:
- String equality checks (`df[df['type'] == 'Movie']`)
- `groupby` — `'Movie'` and `' Movie'` form two separate groups.
- Joins on title or country columns.

We apply `.str.strip()` to all object (string) columns.

**What we do NOT do:** We do NOT strip `description` or `cast` more aggressively (e.g., remove internal double spaces) — that risks altering meaningful text.

In [10]:
str_cols = df.select_dtypes(include='object').columns
print('String columns to strip:', str_cols.tolist())

df[str_cols] = df[str_cols].apply(lambda col: col.str.strip())
print('Done.')

String columns to strip: ['show_id', 'type', 'title', 'director', 'cast', 'country', 'rating', 'duration', 'listed_in', 'description']
Done.


---
## Step 6 — Standardise / Normalise Values

Two targeted fixes:

### 6a. `rating` — Unify `'NR'` and `'UR'` → `'Not Rated'`

**Why:** `NR` (Not Rated) and `UR` (Unrated) mean the same thing in practice. Having three labels for the same concept inflates cardinality and confuses filters.

**What we do NOT do:** We do NOT collapse TV ratings into film ratings (e.g., `TV-MA` ≠ `R`) — they are from different rating systems.

### 6b. `type` — Verify consistent casing

**Why:** Just a sanity check — `'movie'` vs `'Movie'` would break groupby. The dataset only has two clean values so no change needed; we document the check.

In [11]:
# 6a — Unify NR / UR → Not Rated
print('Before:', df['rating'].value_counts(dropna=False).to_dict())

df['rating'] = df['rating'].replace({'NR': 'Not Rated', 'UR': 'Not Rated'})

print('\nAfter:', df['rating'].value_counts(dropna=False).to_dict())

Before: {'TV-MA': 3207, 'TV-14': 2160, 'TV-PG': 863, 'R': 799, 'PG-13': 490, 'TV-Y7': 334, 'TV-Y': 307, 'PG': 287, 'TV-G': 220, 'NR': 80, 'G': 41, 'Not Rated': 7, 'TV-Y7-FV': 6, 'NC-17': 3, 'UR': 3}

After: {'TV-MA': 3207, 'TV-14': 2160, 'TV-PG': 863, 'R': 799, 'PG-13': 490, 'TV-Y7': 334, 'TV-Y': 307, 'PG': 287, 'TV-G': 220, 'Not Rated': 90, 'G': 41, 'TV-Y7-FV': 6, 'NC-17': 3}


In [12]:
# 6b — Verify type column casing
print('Unique values in `type`:', df['type'].unique())
# Only ['Movie', 'TV Show'] — no case normalisation needed.

Unique values in `type`: ['Movie' 'TV Show']


---
## Step 7 — Final Verification & Export

One last audit before we save:
- Confirm shape is unchanged (no rows were dropped)
- Confirm null counts
- Confirm dtypes
- Quick sanity sample

In [13]:
print('=== Final Shape ===')
print(df.shape)   # should still be (8807, 12)

print()
print('=== Final Null Counts ===')
print(df.isnull().sum())

print()
print('=== Final dtypes ===')
print(df.dtypes)

=== Final Shape ===
(8807, 12)

=== Final Null Counts ===
show_id          0
type             0
title            0
director         0
cast             0
country          0
date_added      98
release_year     0
rating           0
duration         0
listed_in        0
description      0
dtype: int64

=== Final dtypes ===
show_id                 object
type                    object
title                   object
director                object
cast                    object
country                 object
date_added      datetime64[ns]
release_year             int64
rating                  object
duration                object
listed_in               object
description             object
dtype: object


In [ ]:
# Spot-check the previously broken Louis C.K. rows
df[df['title'].str.contains('Louis C.K.', na=False)][['title','type','rating','duration','date_added']]

,title,type,rating,duration,date_added
5541,Louis C.K. 2017,Movie,Not Rated,74 min,2017-04-04
5794,Louis C.K.: Hilarious,Movie,Not Rated,84 min,2016-09-16
5813,Louis C.K.: Live at the Comedy Store,Movie,Not Rated,66 min,2016-08-15


In [15]:
df.to_csv(OUTPUT_PATH, index=False)
print(f'Clean dataset saved to: {OUTPUT_PATH}')

Clean dataset saved to: netflix_titles_clean.csv


---
## Summary of All Steps

| Step | Action | Why | Why NOT alternative |
|---|---|---|---|
| 1 | Load & profile | Baseline before any change | — |
| 2 | Fix rating↔duration shift | 3 rows had duration in rating col | Did NOT drop rows — data is valid |
| 3 | Fill nulls column-by-column | Preserve rows; use meaningful sentinels | Did NOT drop or impute with mode/mean |
| 4 | Parse `date_added` to datetime | Enable chronological ops | Did NOT split `duration` — that's feature eng. |
| 5 | Strip whitespace from all str cols | Prevent invisible groupby splits | Did NOT normalise internal spaces in free text |
| 6 | Unify NR/UR → Not Rated | Reduce spurious cardinality | Did NOT collapse TV vs Film rating systems |
| 7 | Final audit + export | Confirm nothing broken | — |

**Rows in:** 8,807 → **Rows out:** 8,807 ✅  
**Columns in:** 12 → **Columns out:** 12 ✅

In [16]:
df.isnull().sum()

show_id          0
type             0
title            0
director         0
cast             0
country          0
date_added      98
release_year     0
rating           0
duration         0
listed_in        0
description      0
dtype: int64